In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [10]:
from src.utils.hadoop_config import configure_windows_hadoop
from src.utils.spark_utils import initialize_spark
from src.utils.parquet_io import read_parquet_data
from src.schemas.silver.dim_question_schema import build_dim_question_schema
from src.schemas.silver.dim_exam_schema import build_dim_exam_schema
from pyspark.sql import functions as F

In [5]:
configure_windows_hadoop()

In [6]:
spark = initialize_spark("jup-read-silver")

In [7]:
SILVER_DIR = os.path.join("..", "data", "silver", "topik_documents")

In [8]:
df_question = read_parquet_data(spark, os.path.join(SILVER_DIR, "dim_question"), build_dim_question_schema())

In [11]:
df_exam = read_parquet_data(spark, os.path.join(SILVER_DIR, "dim_exam"), build_dim_exam_schema())

In [12]:
df_qe = df_question.join(F.broadcast(df_exam),on="exam_id",how="inner")

In [18]:
df_qe.filter(F.col("exam_edition")=="102nd").orderBy(F.col("page_number")).show(10)

+--------------------+--------------------+-----------+--------------------+---------------------------------+------------------------------+------------------------------+---------------------------+---------+----------------+----------+--------------------+------------+-----------+-----------+------------+
|             exam_id|         question_id|page_number|       question_type|                raw_question_text|             question_stimulus|     question_vocabulary_items|  question_grammar_patterns|    topic|difficulty_level|confidence|           exam_name|exam_edition|exam_number|topik_level|exam_section|
+--------------------+--------------------+-----------+--------------------+---------------------------------+------------------------------+------------------------------+---------------------------+---------+----------------+----------+--------------------+------------+-----------+-----------+------------+
|db462efc-0f95-415...|d35dfcdd356362f25...|          5|reading compreh